# Training on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diego-dupuy/Clasificador-Neumonia-Pediatrica/blob/main/notebooks/train_colab.ipynb)

This notebook only **runs** the code in `src/` on a GPU: it downloads the dataset, trains the three
experiments, evaluates them and generates the Grad-CAM figures. All the logic lives in the `.py` files.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Kaggle key: create a token in kaggle.com → *Settings → API*, then in Colab click the 🔑 icon (left bar),
   add a secret called `KAGGLE_API_TOKEN` with the token as value and enable *Notebook access*.

Results are copied to Google Drive after each experiment. If the session disconnects, run everything
again: finished experiments are restored from Drive and skipped.

In [ ]:
# 1) Get the code: clone the repository and move into it
import os
if not os.path.exists("Clasificador-Neumonia-Pediatrica"):
    !git clone -q https://github.com/diego-dupuy/Clasificador-Neumonia-Pediatrica
os.chdir("Clasificador-Neumonia-Pediatrica")

# The only library not pre-installed in Colab
!pip install -q grad-cam

In [ ]:
# 2) Download the dataset from Kaggle.
# The key is read from Colab secrets, so it never appears in the code.
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia
!unzip -q -n chest-xray-pneumonia.zip -d data   # -n: do not overwrite if already extracted
!ls data/chest_xray                              # expected: test  train  val

In [ ]:
# 3) Google Drive: keep results safe if the Colab session disconnects
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RESULTS = "/content/drive/MyDrive/neumonia_results"

# Restore experiments finished in a previous session (if any)
if os.path.exists(DRIVE_RESULTS):
    !mkdir -p results && cp -r "{DRIVE_RESULTS}/." results/
    print("restored:", sorted(os.listdir("results")))

## What the data augmentation does
Each training epoch sees a slightly different version of every X-ray (random crop/zoom, small rotation,
brightness and contrast), so the network cannot rely on borders or corners always being in the same place.
Below: one training image and 5 random augmented versions.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from src.dataset import train_transform

path = sorted(os.listdir("data/chest_xray/train/PNEUMONIA"))[0]
image = Image.open(f"data/chest_xray/train/PNEUMONIA/{path}").convert("RGB")

# First 3 steps of the training transform (crop, rotation, brightness/contrast),
# without tensor conversion and normalisation so the result can be displayed
augment = transforms.Compose(train_transform(True).transforms[:3])

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
axes[0].imshow(image.resize((224, 224)), cmap="gray"); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(augment(image)); ax.set_title("augmented")   # a new random version each time
for ax in axes: ax.axis("off")
plt.show()

## Experiments
Each experiment adds **one** change to the previous one, so the effect of each change can be measured.

| Name | What changes |
|---|---|
| `baseline` | no class weights, no augmentation (reference) |
| `weighted` | + class weights in the loss, to compensate the imbalance (≈3 pneumonia per normal X-ray) |
| `weighted_aug` | + data augmentation |

Each one takes ~10-15 min on a T4. Training stops early if the validation loss does not improve for 4 epochs.

In [ ]:
def run_experiment(name):
    """Train + evaluate one experiment, then copy its results to Drive.
    Skipped if it was already finished (results restored from Drive)."""
    if os.path.exists(f"results/{name}/metrics.json"):
        print(f"{name}: already finished, skipping")
        return
    !python -m src.train --experiment {name}       # trains and saves results/<name>/model.pth
    !python -m src.evaluate --experiment {name}    # metrics, confusion matrix, training curves
    !mkdir -p "{DRIVE_RESULTS}" && cp -r "results/{name}" "{DRIVE_RESULTS}/"

In [ ]:
run_experiment("baseline")

In [ ]:
run_experiment("weighted")

In [ ]:
run_experiment("weighted_aug")

## Comparison
Columns without prefix are measured on the **test set** (624 X-rays). Columns starting with `val_` are
measured on the **validation set**: the final model is chosen with these, and the test set is only reported.

In [ ]:
!python -m src.evaluate --compare

In [ ]:
# Confusion matrices (top) and training curves (bottom) of the three experiments
names = ["baseline", "weighted", "weighted_aug"]
for figure in ["confusion_matrix", "training_curves"]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, name in zip(axes, names):
        ax.imshow(plt.imread(f"results/{name}/{figure}.png")); ax.axis("off")
    plt.tight_layout(); plt.show()

## Grad-CAM: where does the model look?
`central_attention` = share of the heat map inside the central part of the image (15 % margin removed on
each side). If attention were spread evenly it would be ≈ 0.49; higher = more focused on the centre (lungs),
lower = more on borders and corners. Compared for the reference model and the model with augmentation.

In [ ]:
for name in ["baseline", "weighted_aug"]:
    !python -m src.gradcam --experiment {name}
    !cp -r "results/{name}" "{DRIVE_RESULTS}/"     # save the Grad-CAM outputs too

In [ ]:
# Grid of false negatives (pneumonia X-rays predicted as normal) with their heat maps
from IPython.display import Image as Show, display
for name in ["baseline", "weighted_aug"]:
    path = f"results/{name}/gradcam_false_negatives.png"
    if os.path.exists(path):
        display(Show(path))